这个文件中提供了几种实现线性回归的程序
y = np.dot(X, W) + b
1.基于numpy实现线性回归
2.基于torch实现线性回归
3.用torch.nn表示模型
4.使用DataLoader，分批次训练

In [113]:
import numpy as np
import torch
# 设置随机种子以保证结果可重复
torch.manual_seed(0)
np.random.seed(0)


1. 基于numpy实现线性回归
y = np.dot(X, W) + b

In [5]:
import numpy as np

# 定义数据集
X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = np.array([[0.0], [1.0], [0.0], [1.0]])
#y = np.array([[0.0], [1.0], [0.0], [1.0]])#TODO: 线性回归无法解决XOR问题

# 设置超参数
learning_rate = 0.1 #TODO: 0.01, 0.001, ...
num_epochs = 2 #TODO: 5, 10, 100, 1000, 10000, ...

# 初始化权重和偏置
W = np.zeros((2, 1))  # 输入到输出的权重
b = np.zeros((1, 1))  # 输入到输出的偏置

# 训练模型
for epoch in range(num_epochs):
    # 前向传播
    y_pred = np.dot(X, W) + b
     
    # 计算损失
    loss = np.mean((y_pred - y) ** 2)
    
    # 反向传播
    dw = 2*np.mean((y_pred - y)*X, axis=0, keepdims=True).T
    db = 2*np.mean((y_pred - y), axis=0, keepdims=True)
    
    # 更新权重和偏置
    W -= learning_rate * dw
    b -= learning_rate * db
    
    # 打印每个epoch的损失
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss:.4f}')
# 输出模型
print(f"W:{W.flatten()},b:{b.item():.9f}")

# 测试模型
predicted = np.dot(X, W) + b
print("dtype:",predicted.dtype)
print('Predicted:', predicted.flatten())
print('True:', y.flatten())



W:[0.08   0.1775],b:0.165000000
dtype: float64
Predicted: [0.165  0.3425 0.245  0.4225]
True: [0. 1. 0. 1.]


2.基于torch实现线性回归
y = np.dot(X, W) + b

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
type=torch.float64
#device=torch.device("mps")
device=torch.device("cpu")
# 定义数据集
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]],dtype=type)
y = torch.tensor([[0.0], [1.0], [0.0], [1.0]],dtype=type)

# 设置超参数
learning_rate = 0.1
num_epochs = 2

# 初始化权重和偏置
W = torch.zeros((2, 1),dtype=type, requires_grad=True)  # 输入到输出的权重
b = torch.zeros((1, 1),dtype=type, requires_grad=True)  # 输入到输出的偏置

# 定义损失函数
criterion = nn.MSELoss()

# 定义优化器
optimizer = optim.SGD([W, b], lr=learning_rate)

# 训练模型
for epoch in range(num_epochs):
    # 前向传播
    y_pred = torch.mm(X, W) + b
        
     # 计算损失
    loss = criterion(y_pred, y)
        
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
        
    # 更新权重和偏置
    optimizer.step()
    
    # 打印每个epoch的损失
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 输出模型
print(f"W: {W.flatten()}, b: {b.item():.9f}")

# 测试模型
with torch.no_grad():
    predicted = torch.mm(X, W) + b
    print('Predicted:', predicted.numpy().flatten())
    print('True:', y.flatten())


W: tensor([0.0800, 0.1775], dtype=torch.float64, grad_fn=<ViewBackward0>), b: 0.165000000
Predicted: [0.165  0.3425 0.245  0.4225]
True: tensor([0., 1., 0., 1.], dtype=torch.float64)


3.用torch.nn表示模型

In [163]:
import torch
import torch.nn as nn
import torch.optim as optim

# 定义数据集
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=torch.float32)
y = torch.tensor([[0.0], [1.0], [1.0], [1.0]], dtype=torch.float32)
#y = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)

# 设置超参数
learning_rate = 0.1
num_epochs = 500

# 定义模型
class Perceptron(nn.Module):
    def __init__(self, input_size, output_size=1):
        super(Perceptron, self).__init__()
        self.linear = nn.Linear(input_size, output_size,dtype=torch.float32)       
        # 手动将权重和偏置初始化为零
        nn.init.constant_(self.linear.weight, 0)
        nn.init.constant_(self.linear.bias, 0)   
            
    def forward(self, x):
        return self.linear(x)

class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size=4, output_size=1):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # 输入到隐藏层
        self.relu = nn.ReLU()  # ReLU激活函数
        self.fc2 = nn.Linear(hidden_size, output_size)  # 隐藏层到输出
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# 初始化模型、损失函数和优化器
model = Perceptron(input_size=2)
criterion = nn.MSELoss()#损失函数的输入是模型输出和参考答案
optimizer = optim.SGD(model.parameters(), lr=learning_rate)#优化器和模型参数绑定

# 训练模型
for epoch in range(num_epochs):
    # 前向传播
    y_pred = model(X)
    
    # 计算损失
    loss = criterion(y_pred, y)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    
    # 更新权重和偏置
    optimizer.step()
    
    # 打印每个epoch的损失
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 输出模型
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.data.flatten()}")

# 测试模型
with torch.no_grad():
    predicted = model(X)
    print('Predicted:', predicted.numpy().flatten())
    print('True:', y.numpy().flatten())


Epoch [100/500], Loss: 0.0625
Epoch [200/500], Loss: 0.0625
Epoch [300/500], Loss: 0.0625
Epoch [400/500], Loss: 0.0625
Epoch [500/500], Loss: 0.0625
linear.weight: tensor([0.5000, 0.5000])
linear.bias: tensor([0.2500])
Predicted: [0.2500004  0.75000006 0.75000006 1.2499998 ]
True: [0. 1. 1. 1.]


4.使用DataLoader，分批次训练
TODO：分批次训练，乱序
TODO：使用Adam算参数
TODO: 使用GPU
TODO：可视化


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
device=torch.device("cpu")

# 定义数据集
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]],dtype=torch.float32)
X = X.to(device)
y = torch.tensor([[0.0], [1.0], [0.0], [1.0]],  dtype=torch.float32)
y = y.to(device)

# 创建TensorDataset和DataLoader
dataset = TensorDataset(X, y)
#print(dataset[0])
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)#TODO：修改batch_size大小,把数据打乱

# 设置超参数
learning_rate = 0.1
num_epochs = 500

# 定义模型
class Perceptron(nn.Module):
    def __init__(self, input_size, output_size):
        super(Perceptron, self).__init__()
        self.linear = nn.Linear(input_size, output_size,dtype=torch.float32)
        # 手动将权重和偏置初始化为零
        #nn.init.constant_(self.linear.weight, 0)
        #nn.init.constant_(self.linear.bias, 0)
        
    def forward(self, x):
        return self.linear(x)

# 初始化模型、损失函数和优化器
model = Perceptron(input_size=2, output_size=1)
model.to(device)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# 训练模型
for epoch in range(num_epochs):
    for inputs, targets in dataloader:
        # 前向传播
        y_pred = model(inputs)
        
        # 计算损失
        loss = criterion(y_pred, targets)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        
        # 更新权重和偏置
        optimizer.step()
    
    # 打印每个epoch的损失
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 输出模型
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.data.flatten()}")

# 测试模型
with torch.no_grad():
    predicted = model(X)
    #predicted = (predicted > 0.5).int()
    print('Predicted:', predicted.flatten())#cpu().numpy().
    print('True:', y.flatten())



Epoch [100/500], Loss: 0.0000
Epoch [200/500], Loss: 0.0000
Epoch [300/500], Loss: 0.0000
Epoch [400/500], Loss: 0.0000
Epoch [500/500], Loss: 0.0000
linear.weight: tensor([7.0189e-09, 1.0000e+00])
linear.bias: tensor([-4.8118e-07])
Predicted: tensor([-4.8118e-07,  1.0000e+00, -4.7416e-07,  1.0000e+00])
True: tensor([0., 1., 0., 1.])


分类问题
TODO:把二分类问题拓宽到多类问题

In [181]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)
device=torch.device("mps")

# 定义数据集
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=torch.float32)
X=X.to(device)
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]], dtype=torch.float32)  # XOR 输出
y=y.to(device)
# 设置超参数
learning_rate = 0.01
num_epochs = 500

# 定义模型
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # 输入到隐藏层的全连接层
        self.sigmoid = nn.Sigmoid()                    # 隐藏层的激活函数
        self.fc2 = nn.Linear(hidden_size, output_size) # 隐藏层到输出层的全连接层
        self.output_activation = nn.Sigmoid()          # 输出层的激活函数

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)
        out = self.output_activation(out)  # 输出层使用Sigmoid激活函数
        return out

# 初始化模型、损失函数和优化器
input_size = 2
hidden_size = 4
output_size = 1

model = MLP(input_size, hidden_size, output_size)
model.to(device)
criterion = nn.BCELoss()  # 使用二元交叉熵损失函数
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 训练模型
for epoch in range(num_epochs):
    # 前向传播
    y_pred = model(X)
    
    # 计算损失
    loss = criterion(y_pred, y)
    
    # 反向传播和优化
    optimizer.zero_grad()  # 清零梯度
    loss.backward()        # 反向传播计算梯度
    optimizer.step()       # 更新权重和偏置
    
    # 打印每个epoch的损失
    if (epoch+1) % 1000 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 输出模型参数
print("Model parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.data.flatten()}")

# 测试模型
with torch.no_grad():
    predicted = model(X)
    #predicted = (predicted > 0.5).int()

    print('Predicted:', predicted.flatten())
    print('True:', y.flatten())



Model parameters:
fc1.weight: tensor([ 3.4803,  3.5148, -5.1591, -5.0668,  3.6400,  3.6506,  5.3835,  5.7447],
       device='mps:0')
fc1.bias: tensor([-5.2024,  2.0577, -5.4189, -2.3055], device='mps:0')
fc2.weight: tensor([-4.7806, -4.7296, -4.5874,  4.4629], device='mps:0')
fc2.bias: tensor([0.5173], device='mps:0')
Predicted: tensor([0.0349, 0.9610, 0.9606, 0.0433], device='mps:0')
True: tensor([0., 1., 1., 0.], device='mps:0')
